# 07 — Composite TreeSHAP for Full Hybrid Model

Weighted combination of TreeSHAP from **XGBoost**, **LightGBM**, and **RandomForest**, weighted by the Ridge meta-learner coefficients.

**How it works:**
```
final_pred = w0*pred_ridge + w1*pred_xgb + w2*pred_lgb + w3*pred_rf
```

Since SHAP is additive, we can compute:
```
composite_SHAP ≈ w1*SHAP_xgb + w2*SHAP_lgb + w3*SHAP_rf
```

**Prereqs:** Run **02_hybrid_ensemble.ipynb** so `03_ml_layer_hybrid/artifacts/hybrid_cluster_bundle.joblib` exists.

**Trade-offs:**
- **Pros:** Fast (~5ms per row), captures all tree-based models
- **Cons:** Ignores Ridge component (~5% of signal), assumes linear meta-weights

**Outputs:** `03_ml_layer_hybrid/artifacts/hybrid_xai/composite_treeshap_*.joblib`

In [13]:
%pip install -q numpy pandas scikit-learn xgboost lightgbm shap matplotlib joblib

Note: you may need to restart the kernel to use updated packages.


In [14]:
import importlib
import json
import time
import warnings
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.metrics import mean_absolute_percentage_error

warnings.filterwarnings("ignore")

import sys

_HERE = Path.cwd().resolve()


def _repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "feature_data").is_dir() or (p / "hf_data").is_dir():
            return p
    return start.parent


REPO_ROOT = _repo_root(_HERE)

_FEATURE_OUTPUT_CANDIDATES = [
    REPO_ROOT / "hf_data" / "02_feature_layer" / "training" / "outputs",
    REPO_ROOT / "data" / "feature_data" / "02_feature_layer" / "training" / "outputs",
]


def _feature_outputs_dir() -> Path:
    for d in _FEATURE_OUTPUT_CANDIDATES:
        if d.is_dir() and any(d.glob("hdb_feature_table_*.csv")):
            return d
    tried = "\n  ".join(str(d) for d in _FEATURE_OUTPUT_CANDIDATES)
    raise FileNotFoundError(
        "No hdb_feature_table_*.csv found. Run notebooks/00_download_data_from_HF.ipynb "
        "or place CSVs under one of:\n  " + tried
    )


def _hybrid_ml_dir(repo: Path) -> Path:
    candidates = [
        repo / "notebooks" / "03_ml_layer_hybrid",
        repo / "03_ml_layer_hybrid",
    ]
    for d in candidates:
        if (d / "yc_hybrid_inference.py").is_file():
            return d
    raise FileNotFoundError(
        "yc_hybrid_inference.py not found. Expected under notebooks/03_ml_layer_hybrid/."
    )


HYBRID_DIR = _hybrid_ml_dir(REPO_ROOT)
if str(HYBRID_DIR) not in sys.path:
    sys.path.insert(0, str(HYBRID_DIR))

import yc_hybrid_inference
importlib.reload(yc_hybrid_inference)
from yc_hybrid_inference import predict_price, load_bundle

HF_DATA_ROOT = _feature_outputs_dir()


def _latest_feature_snapshot_date(root: Path) -> str:
    tables = sorted(root.glob("hdb_feature_table_*.csv"))
    if not tables:
        raise FileNotFoundError(f"No hdb_feature_table_*.csv under {root}")
    return tables[-1].stem.split("_")[-1]


_snap = _latest_feature_snapshot_date(HF_DATA_ROOT)
all_path = HF_DATA_ROOT / f"hdb_feature_table_{_snap}.csv"

HERE = REPO_ROOT
ART = HYBRID_DIR / "artifacts"
OUT_DIR = ART / "hybrid_xai"
OUT_DIR.mkdir(parents=True, exist_ok=True)
BUNDLE_PATH = ART / "hybrid_cluster_bundle.joblib"

TARGET = "resale_price"
YEAR_COL = "transaction_year"

print("HF_DATA_ROOT:", HF_DATA_ROOT)
print("Feature table:", all_path.name)
print("Artifacts:", ART)
print("OUT_DIR:", OUT_DIR)

HF_DATA_ROOT: /Users/bhuvesh/Documents/PropertyLens/data/feature_data/02_feature_layer/training/outputs
Feature table: hdb_feature_table_20260412.csv
Artifacts: /Users/bhuvesh/Documents/PropertyLens/notebooks/03_ml_layer_hybrid/artifacts
OUT_DIR: /Users/bhuvesh/Documents/PropertyLens/notebooks/03_ml_layer_hybrid/artifacts/hybrid_xai


## 1 — Load data and hybrid bundle

In [15]:
df = pd.read_csv(all_path)
df = df.sort_values([YEAR_COL, "address_key"], kind="mergesort").reset_index(drop=True)

bundle = joblib.load(BUNDLE_PATH)
FEATURE_COLS = bundle["feature_columns"]
N_CLUSTERS = int(bundle["n_clusters"])
CLUSTER_COLS = bundle["cluster_cols"]

train_mask = df[YEAR_COL] < 2024
val_mask = df[YEAR_COL] == 2024
test_mask = df[YEAR_COL] >= 2025

X_all = df[FEATURE_COLS].fillna(0).astype(float)
y_all = df[TARGET].astype(float)

X_train = X_all.loc[train_mask].values
y_train = y_all.loc[train_mask].values
X_test = X_all.loc[test_mask].values
y_test = y_all.loc[test_mask].values

print("Train rows:", len(X_train), "| Test rows:", len(X_test), "| Features:", len(FEATURE_COLS))
print("Clusters:", N_CLUSTERS)

Train rows: 205930 | Test rows: 29266 | Features: 70
Clusters: 4


## 2 — Build TreeExplainers for each model per cluster

We create `shap.TreeExplainer` for XGBoost, LightGBM, and RandomForest for each cluster, plus the global models for fallback.

In [16]:
import xgboost as xgb


class XGBPredContribsExplainer:
    """Drop-in replacement for ``shap.TreeExplainer`` for XGBoost models.

    Uses XGBoost's native ``Booster.predict(..., pred_contribs=True)`` so it sidesteps
    ``shap.TreeExplainer``'s parse of ``learner_model_param["base_score"]`` — which raises
    ``ValueError: could not convert string to float: '[5.692308E5]'`` on boosters saved by
    newer XGBoost versions.

    ``base + sum(shap_i)`` equals the XGB component's prediction (same scope a
    TreeExplainer on the XGB booster would have given). Matches ``backend/shap_local.py``.
    """

    def __init__(self, model, feature_names):
        self.model = model
        self.booster = model.get_booster()
        self.feature_names = list(feature_names)
        zero_row = pd.DataFrame(np.zeros((1, len(self.feature_names))), columns=self.feature_names)
        contribs = np.asarray(
            self.booster.predict(xgb.DMatrix(zero_row, feature_names=self.feature_names), pred_contribs=True)
        ).ravel()
        self.expected_value = float(contribs[-1])

    def __call__(self, X):
        X_arr = np.asarray(X, dtype=float)
        if X_arr.ndim == 1:
            X_arr = X_arr.reshape(1, -1)
        df = pd.DataFrame(X_arr, columns=self.feature_names)
        contribs = np.asarray(
            self.booster.predict(xgb.DMatrix(df, feature_names=self.feature_names), pred_contribs=True)
        )
        values = contribs[:, :-1]
        base_values = np.full(len(X_arr), self.expected_value)
        return shap.Explanation(
            values=values,
            base_values=base_values,
            data=X_arr,
            feature_names=self.feature_names,
        )


print("Building TreeExplainers for each cluster...")
t0 = time.time()


def safe_tree_explainer(model, model_type="unknown"):
    """Build a SHAP explainer. For XGBoost, bypass ``shap.TreeExplainer`` entirely
    and use native ``pred_contribs`` — matches ``backend/shap_local.py`` and avoids
    the base_score string-encoding issue on newer XGBoost versions."""
    if model_type == "xgb":
        return XGBPredContribsExplainer(model, FEATURE_COLS)
    return shap.TreeExplainer(model)


cluster_bundles = bundle["cluster_bundles"]
cluster_explainers = {}
fallback_cluster_ids = []

for k in range(N_CLUSTERS):
    cb = cluster_bundles.get(k, cluster_bundles.get(str(k), {}))
    if cb.get("fallback"):
        fallback_cluster_ids.append(k)
        print(f"  Cluster {k}: fallback → will use global explainers")
    else:
        cluster_explainers[k] = {
            "xgb": safe_tree_explainer(cb["xgb"], "xgb"),
            "lgb": safe_tree_explainer(cb["lgb"], "lgb"),
            "rf": safe_tree_explainer(cb["rf"], "rf"),
            "meta_coefs": cb["meta"].coef_,
            "meta_intercept": float(cb["meta"].intercept_),
        }
        print(f"  Cluster {k}: TreeExplainers built ✓")
        print(f"    Meta coefs: ridge={cb['meta'].coef_[0]:.4f}, xgb={cb['meta'].coef_[1]:.4f}, "
              f"lgb={cb['meta'].coef_[2]:.4f}, rf={cb['meta'].coef_[3]:.4f}")

# Global explainers for fallback
gm = bundle["global_models"]
global_explainers = {
    "xgb": safe_tree_explainer(gm["xgb"], "xgb"),
    "lgb": safe_tree_explainer(gm["lgb"], "lgb"),
    "rf": safe_tree_explainer(gm["rf"], "rf"),
    "meta_coefs": gm["meta"].coef_,
    "meta_intercept": float(gm["meta"].intercept_),
}
print(f"\nGlobal explainers built ✓")
print(f"  Meta coefs: ridge={gm['meta'].coef_[0]:.4f}, xgb={gm['meta'].coef_[1]:.4f}, "
      f"lgb={gm['meta'].coef_[2]:.4f}, rf={gm['meta'].coef_[3]:.4f}")

print(f"\nTotal build time: {time.time() - t0:.1f}s")

Building TreeExplainers for each cluster...
  Cluster 0: TreeExplainers built ✓
    Meta coefs: ridge=0.0967, xgb=0.6991, lgb=0.4905, rf=-0.2057
  Cluster 1: TreeExplainers built ✓
    Meta coefs: ridge=-0.0113, xgb=0.9709, lgb=0.3581, rf=-0.2137
  Cluster 2: TreeExplainers built ✓
    Meta coefs: ridge=0.2177, xgb=1.1301, lgb=0.0454, rf=-0.2493
  Cluster 3: TreeExplainers built ✓
    Meta coefs: ridge=0.0661, xgb=0.2189, lgb=0.9577, rf=-0.1985

Global explainers built ✓
  Meta coefs: ridge=-0.0301, xgb=0.9479, lgb=0.2926, rf=-0.1353

Total build time: 2.5s


## 3 — CompositeTreeSHAPExplainer class

A reusable class that:
1. Routes each row to its cluster (via K-Means)
2. Computes TreeSHAP for XGB, LGB, RF
3. Weights by meta-learner coefficients
4. Returns composite SHAP values

In [17]:
class CompositeTreeSHAPExplainer:
    """
    Full hybrid SHAP explainer using composite TreeSHAP.
    
    Combines SHAP values from XGBoost, LightGBM, and RandomForest,
    weighted by the Ridge meta-learner coefficients.
    """
    
    def __init__(self, bundle, cluster_explainers, global_explainers):
        self.bundle = bundle
        self.n_clusters = bundle["n_clusters"]
        self.cluster_scaler = bundle["cluster_scaler"]
        self.kmeans = bundle["kmeans"]
        self.cluster_cols = bundle["cluster_cols"]
        self.feature_cols = bundle["feature_columns"]
        self.cluster_bundles = bundle["cluster_bundles"]
        self.cluster_explainers = cluster_explainers
        self.global_explainers = global_explainers
    
    def _get_cluster(self, row):
        """Route a single row to its cluster."""
        idx_feat = [self.feature_cols.index(c) for c in self.cluster_cols]
        Xc = row[idx_feat].reshape(1, -1)
        Xc_scaled = self.cluster_scaler.transform(Xc)
        return int(self.kmeans.predict(Xc_scaled)[0])
    
    def _get_expected_value(self, exp_dict):
        """Extract expected value from explainer dict."""
        base_xgb = exp_dict["xgb"].expected_value
        base_lgb = exp_dict["lgb"].expected_value
        base_rf = exp_dict["rf"].expected_value
        
        # Handle array-like expected values
        if hasattr(base_xgb, '__iter__'):
            base_xgb = float(np.array(base_xgb).flatten()[0])
        if hasattr(base_lgb, '__iter__'):
            base_lgb = float(np.array(base_lgb).flatten()[0])
        if hasattr(base_rf, '__iter__'):
            base_rf = float(np.array(base_rf).flatten()[0])
        
        coefs = exp_dict["meta_coefs"]
        # Weighted base value: w1*base_xgb + w2*base_lgb + w3*base_rf + intercept
        return coefs[1] * base_xgb + coefs[2] * base_lgb + coefs[3] * base_rf + exp_dict["meta_intercept"]
    
    def shap_values(self, X):
        """
        Compute composite SHAP values for each row.
        
        Args:
            X: Feature matrix (n_samples, n_features)
            
        Returns:
            shap.Explanation object with composite SHAP values
        """
        X = np.asarray(X, dtype=float)
        if X.ndim == 1:
            X = X.reshape(1, -1)
        
        all_shap = np.zeros_like(X)
        base_values = np.zeros(len(X))
        cluster_assignments = np.zeros(len(X), dtype=int)
        
        for i in range(len(X)):
            row = X[i:i+1]
            k = self._get_cluster(row.flatten())
            cluster_assignments[i] = k
            
            cb = self.cluster_bundles.get(k, self.cluster_bundles.get(str(k), {}))
            
            if cb.get("fallback") or k not in self.cluster_explainers:
                exp = self.global_explainers
            else:
                exp = self.cluster_explainers[k]
            
            # Get SHAP from each tree model
            sv_xgb = exp["xgb"](row).values[0]
            sv_lgb = exp["lgb"](row).values[0]
            sv_rf = exp["rf"](row).values[0]
            
            # Weighted by meta-learner coefficients (indices: 0=ridge, 1=xgb, 2=lgb, 3=rf)
            coefs = exp["meta_coefs"]
            composite = coefs[1] * sv_xgb + coefs[2] * sv_lgb + coefs[3] * sv_rf
            
            all_shap[i] = composite
            base_values[i] = self._get_expected_value(exp)
        
        return shap.Explanation(
            values=all_shap,
            base_values=base_values,
            data=X,
            feature_names=self.feature_cols,
        ), cluster_assignments
    
    def __call__(self, X):
        """Allow calling the explainer directly."""
        explanation, _ = self.shap_values(X)
        return explanation


# Create the composite explainer
composite_explainer = CompositeTreeSHAPExplainer(bundle, cluster_explainers, global_explainers)
print("CompositeTreeSHAPExplainer created ✓")

CompositeTreeSHAPExplainer created ✓


## 4 — Compute SHAP values for test samples

Composite TreeSHAP is fast, so we can compute SHAP for many more samples than KernelSHAP.

In [ ]:
# Cluster-stratified sampling across train + test (val dropped — small + overlaps).
# Equal share per cluster keeps small clusters out of noise territory while keeping
# total compute bounded (~2 min on CPU at N=2000, ~66ms/row).
N_SAMPLES = 2000
PER_CLUSTER = N_SAMPLES // N_CLUSTERS

X_pool = np.vstack([X_train, X_test])
y_pool = np.concatenate([y_train, y_test])
print(f"Combined pool: {len(X_pool):,} rows ({len(X_train):,} train + {len(X_test):,} test)")

# Route every row to its cluster in one vectorized pass.
idx_feat = [FEATURE_COLS.index(c) for c in CLUSTER_COLS]
Xc_all = X_pool[:, idx_feat].astype(float)
Xc_scaled = bundle["cluster_scaler"].transform(Xc_all)
pool_cluster_ids = bundle["kmeans"].predict(Xc_scaled).astype(int)

rng = np.random.RandomState(42)
sample_indices = []
print("\nCluster-stratified sampling:")
for k in range(N_CLUSTERS):
    cand = np.where(pool_cluster_ids == k)[0]
    take = min(PER_CLUSTER, len(cand))
    print(f"  Cluster {k}: {len(cand):,} available → drawing {take:,}")
    if take == 0:
        continue
    sample_indices.append(rng.choice(cand, take, replace=False))

sample_indices = np.concatenate(sample_indices)
rng.shuffle(sample_indices)
X_sample = X_pool[sample_indices]
y_sample = y_pool[sample_indices]
N_SAMPLES = len(X_sample)

print(f"\nComputing Composite TreeSHAP values for {N_SAMPLES:,} samples...")
print("-" * 60)

t0 = time.time()
explanation, cluster_assignments = composite_explainer.shap_values(X_sample)
elapsed = time.time() - t0

shap_values = explanation.values
base_values = explanation.base_values

print(f"\nCompleted in {elapsed:.1f}s ({elapsed*1000/N_SAMPLES:.1f}ms per row)")
print(f"SHAP values shape: {shap_values.shape}")
print(f"\nFinal cluster distribution in sample:")
for k in range(N_CLUSTERS):
    count = (cluster_assignments == k).sum()
    print(f"  Cluster {k}: {count} samples ({count/N_SAMPLES*100:.1f}%)")

## 5 — Validate: SHAP additivity check

Verify that base_value + sum(SHAP) ≈ prediction for each sample.

In [19]:
# Get actual predictions from hybrid model
predictions = predict_price(X_sample, bundle)

# SHAP additivity: base + sum(SHAP) should equal prediction
# Note: This is approximate because we're using tree SHAP weighted by meta coefs,
# not the full hybrid including Ridge
shap_reconstructed = base_values + shap_values.sum(axis=1)

# Compare
errors = predictions - shap_reconstructed
abs_errors = np.abs(errors)

print("SHAP Additivity Check (base + sum(SHAP) vs actual prediction):")
print(f"  Mean absolute difference: ${abs_errors.mean():,.0f}")
print(f"  Median absolute difference: ${np.median(abs_errors):,.0f}")
print(f"  Max absolute difference: ${abs_errors.max():,.0f}")
print(f"  Mean prediction: ${predictions.mean():,.0f}")
print(f"  Relative error: {abs_errors.mean() / predictions.mean() * 100:.2f}%")
print("\nNote: Some difference is expected because we're approximating the full hybrid")
print("      (Ridge component and meta-learner interactions are simplified)")

SHAP Additivity Check (base + sum(SHAP) vs actual prediction):
  Mean absolute difference: $47,447
  Median absolute difference: $41,770
  Max absolute difference: $233,832
  Mean prediction: $652,557
  Relative error: 7.27%

Note: Some difference is expected because we're approximating the full hybrid
      (Ridge component and meta-learner interactions are simplified)


## 6 — Visualizations

### 6.1 Global feature importance

In [20]:
# Global feature importance (mean |SHAP|)
mean_abs_shap = np.abs(shap_values).mean(axis=0)
feature_importance = dict(zip(FEATURE_COLS, mean_abs_shap))
feature_importance = dict(sorted(feature_importance.items(), key=lambda x: x[1], reverse=True))

print("Top 10 features by mean |SHAP| (Composite TreeSHAP):")
for i, (feat, imp) in enumerate(list(feature_importance.items())[:10]):
    print(f"  {i+1:2d}. {feat:35s} ${imp:>12,.0f}")

Top 10 features by mean |SHAP| (Composite TreeSHAP):
   1. transaction_year                    $     114,566
   2. floor_area_sqm                      $      52,951
   3. lease_remaining_years               $      36,569
   4. room_count                          $      24,371
   5. dist_to_highway_m                   $      18,826
   6. level_mid                           $      17,987
   7. mall_weighted_access_3km            $       7,607
   8. dist_to_foodcourt_m                 $       7,217
   9. dist_to_nearest_mall_m              $       5,053
  10. dist_to_mrt_m                       $       4,529


In [21]:
# Summary beeswarm plot
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, feature_names=FEATURE_COLS, show=False, max_display=15)
plt.tight_layout()
plt.savefig(OUT_DIR / "composite_treeshap_summary_beeswarm.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved", OUT_DIR / "composite_treeshap_summary_beeswarm.png")

Saved /Users/bhuvesh/Documents/PropertyLens/notebooks/03_ml_layer_hybrid/artifacts/hybrid_xai/composite_treeshap_summary_beeswarm.png


In [22]:
# Bar chart of mean |SHAP| importance
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, feature_names=FEATURE_COLS, plot_type="bar", show=False, max_display=15)
plt.tight_layout()
plt.savefig(OUT_DIR / "composite_treeshap_summary_bar.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved", OUT_DIR / "composite_treeshap_summary_bar.png")

Saved /Users/bhuvesh/Documents/PropertyLens/notebooks/03_ml_layer_hybrid/artifacts/hybrid_xai/composite_treeshap_summary_bar.png


### 6.2 Waterfall plot (single prediction explanation)

In [23]:
# Waterfall plot for first sample
idx = 0
pred = predictions[idx]
actual = y_sample[idx]

print(f"Sample {idx} (Cluster {cluster_assignments[idx]}):")
print(f"  Actual: ${actual:,.0f}")
print(f"  Predicted: ${pred:,.0f}")
print(f"  Base value: ${base_values[idx]:,.0f}")
print(f"  Sum of SHAP: ${shap_values[idx].sum():,.0f}")
print(f"  Base + SHAP: ${base_values[idx] + shap_values[idx].sum():,.0f}")

fig, ax = plt.subplots(figsize=(10, 8))
shap.waterfall_plot(explanation[idx], show=False, max_display=12)
plt.tight_layout()
plt.savefig(OUT_DIR / "composite_treeshap_waterfall_sample0.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved", OUT_DIR / "composite_treeshap_waterfall_sample0.png")

Sample 0 (Cluster 0):
  Actual: $750,000
  Predicted: $861,985
  Base value: $555,302
  Sum of SHAP: $233,070
  Base + SHAP: $788,372
Saved /Users/bhuvesh/Documents/PropertyLens/notebooks/03_ml_layer_hybrid/artifacts/hybrid_xai/composite_treeshap_waterfall_sample0.png


### 6.3 Per-cluster feature importance

In [24]:
# Per-cluster feature importance
cluster_importance = {}

for k in range(N_CLUSTERS):
    mask = cluster_assignments == k
    if mask.sum() == 0:
        continue
    
    cluster_shap = shap_values[mask]
    mean_abs = np.abs(cluster_shap).mean(axis=0)
    importance = dict(zip(FEATURE_COLS, mean_abs))
    importance = dict(sorted(importance.items(), key=lambda x: x[1], reverse=True))
    cluster_importance[k] = importance
    
    print(f"\nCluster {k} ({mask.sum()} samples) - Top 5 features:")
    for i, (feat, imp) in enumerate(list(importance.items())[:5]):
        print(f"  {i+1}. {feat:30s} ${imp:>10,.0f}")


Cluster 0 (197 samples) - Top 5 features:
  1. transaction_year               $   133,477
  2. floor_area_sqm                 $    43,759
  3. lease_remaining_years          $    34,981
  4. dist_to_highway_m              $    29,525
  5. room_count                     $    26,439

Cluster 1 (162 samples) - Top 5 features:
  1. transaction_year               $    95,980
  2. floor_area_sqm                 $    55,594
  3. lease_remaining_years          $    40,142
  4. room_count                     $    20,742
  5. dist_to_highway_m              $    14,301

Cluster 2 (37 samples) - Top 5 features:
  1. floor_area_sqm                 $   122,489
  2. transaction_year               $   104,709
  3. lease_remaining_years          $    81,510
  4. level_mid                      $    29,993
  5. room_count                     $    22,889

Cluster 3 (104 samples) - Top 5 features:
  1. transaction_year               $   111,202
  2. floor_area_sqm                 $    41,506
  3. room_cou

## 7 — Save artifacts

In [25]:
# Save cluster explainers (the TreeExplainers are already in shap_explainers.joblib)
# Save the composite explainer configuration
composite_config = {
    "n_clusters": N_CLUSTERS,
    "fallback_cluster_ids": fallback_cluster_ids,
    "feature_columns": FEATURE_COLS,
    "cluster_cols": CLUSTER_COLS,
}

# Save per-cluster meta coefficients
meta_coefs_by_cluster = {}
for k in range(N_CLUSTERS):
    cb = cluster_bundles.get(k, cluster_bundles.get(str(k), {}))
    if cb.get("fallback"):
        meta_coefs_by_cluster[str(k)] = {
            "fallback": True,
            "uses_global": True,
            "coefs": global_explainers["meta_coefs"].tolist(),
            "intercept": global_explainers["meta_intercept"],
        }
    else:
        meta_coefs_by_cluster[str(k)] = {
            "fallback": False,
            "coefs": cluster_explainers[k]["meta_coefs"].tolist(),
            "intercept": cluster_explainers[k]["meta_intercept"],
        }

composite_config["meta_coefs_by_cluster"] = meta_coefs_by_cluster
composite_config["global_meta_coefs"] = global_explainers["meta_coefs"].tolist()
composite_config["global_meta_intercept"] = global_explainers["meta_intercept"]

with open(OUT_DIR / "composite_treeshap_config.json", "w") as f:
    json.dump(composite_config, f, indent=2)
print("Saved", OUT_DIR / "composite_treeshap_config.json")

Saved /Users/bhuvesh/Documents/PropertyLens/notebooks/03_ml_layer_hybrid/artifacts/hybrid_xai/composite_treeshap_config.json


In [ ]:
# Save computed SHAP values
composite_shap_payload = {
    "shap_values": shap_values,
    "base_values": base_values,
    "X_sample": X_sample,
    "y_sample": y_sample,
    "sample_indices": sample_indices,
    "cluster_assignments": cluster_assignments,
    "feature_columns": FEATURE_COLS,
}
joblib.dump(composite_shap_payload, OUT_DIR / "composite_treeshap_values.joblib")
print("Saved", OUT_DIR / "composite_treeshap_values.joblib")

# Ridge reconciliation gap — what the tree-only attribution can't explain.
# Surfaced in the JSON so the analytics page can be honest about it.
shap_reconstructed = base_values + shap_values.sum(axis=1)
ridge_gap = predictions - shap_reconstructed
ridge_gap_mean_abs = float(np.abs(ridge_gap).mean())
ridge_gap_pct_of_pred = float(np.abs(ridge_gap).mean() / predictions.mean())

# Save global and per-cluster importance as JSON
composite_importance = {
    "method": "CompositeTreeSHAP",
    "model": "full_hybrid_ensemble",
    "n_samples": int(N_SAMPLES),
    "sample_strategy": "cluster_stratified_train_plus_test",
    "ridge_gap_mean_abs": ridge_gap_mean_abs,
    "ridge_gap_pct_of_pred": ridge_gap_pct_of_pred,
    "global_feature_importance": {k: float(v) for k, v in feature_importance.items()},
    "per_cluster_importance": {
        str(k): {f: float(v) for f, v in imp.items()}
        for k, imp in cluster_importance.items()
    },
}
with open(OUT_DIR / "composite_treeshap_global_importance.json", "w") as f:
    json.dump(composite_importance, f, indent=2)
print("Saved", OUT_DIR / "composite_treeshap_global_importance.json")

# Mirror to backend artifacts dir so the FastAPI /api/analytics/global-shap
# endpoint serves the freshest composite SHAP without a manual copy step.
backend_xai = REPO_ROOT / "data" / "artifacts" / "hybrid_xai"
backend_xai.mkdir(parents=True, exist_ok=True)
with open(backend_xai / "composite_treeshap_global_importance.json", "w") as f:
    json.dump(composite_importance, f, indent=2)
print("Saved", backend_xai / "composite_treeshap_global_importance.json")

In [ ]:
# Cluster profiles — human-readable summary of what each k-means cluster contains.
# Auto-regenerated each run so the analytics page label never drifts from the
# actual model. Labels are derived heuristically from the per-cluster stats; tweak
# the wording in the dict below if a particular cluster needs a clearer story.

profile_feats = [
    "floor_area_sqm",
    "lease_remaining_years",
    "level_mid",
    "room_count",
    "dist_to_mrt_m",
    "resale_price",
]
df_full = pd.read_csv(all_path)
X_full = df_full[FEATURE_COLS].fillna(0).astype(float).values

# Route every row in the full feature table to its cluster (vectorized).
idx_feat_all = [FEATURE_COLS.index(c) for c in CLUSTER_COLS]
Xc_full = X_full[:, idx_feat_all].astype(float)
full_clusters = bundle["kmeans"].predict(bundle["cluster_scaler"].transform(Xc_full))
df_full["_cluster"] = full_clusters

# Per-cluster stats over the full feature table.
stats_df = df_full.groupby("_cluster")[profile_feats].mean().round(1)

# Top-5 town one-hots per cluster.
town_cols_all = [c for c in FEATURE_COLS if c.startswith("town_")]
town_idx = [FEATURE_COLS.index(t) for t in town_cols_all]


def _top_towns(mask):
    sums = X_full[mask][:, town_idx].sum(axis=0)
    pairs = sorted(zip(town_cols_all, sums), key=lambda x: -x[1])
    return [t.replace("town_", "") for t, s in pairs[:5] if s > 0]


def _heuristic_label(s):
    """Tag a cluster from its mean stats — rough, but auto-updates with retraining."""
    sqm = s["floor_area_sqm"]
    lease = s["lease_remaining_years"]
    floor = s["level_mid"]
    mrt = s["dist_to_mrt_m"]
    if mrt > 3000:
        return "Outlying larger flats far from MRT"
    if floor >= 11:
        return "Central high-floor premium flats"
    if sqm >= 100 and lease >= 75:
        return "Newer suburban larger flats"
    if sqm < 85 and lease < 70:
        return "Older compact mature-estate flats"
    return "Mid-segment flats"


def _summary(s):
    return (
        f"~{s['floor_area_sqm']:.0f} sqm · "
        f"~{s['lease_remaining_years']:.0f}yr lease · "
        f"floor ~{s['level_mid']:.0f} · "
        f"{s['dist_to_mrt_m']/1000:.1f}km from MRT"
    )


total_rows = len(df_full)
profiles_clusters = {}
for k in sorted(stats_df.index):
    s = stats_df.loc[k]
    mask = full_clusters == k
    profiles_clusters[str(int(k))] = {
        "label": _heuristic_label(s),
        "summary": _summary(s),
        "top_towns": _top_towns(mask),
        "share": float(mask.sum() / total_rows),
        "n_rows": int(mask.sum()),
        "stats": {
            "floor_area_sqm": float(s["floor_area_sqm"]),
            "lease_remaining_years": float(s["lease_remaining_years"]),
            "level_mid": float(s["level_mid"]),
            "room_count": float(s["room_count"]),
            "dist_to_mrt_m": float(s["dist_to_mrt_m"]),
            "resale_price_mean": float(s["resale_price"]),
        },
    }

cluster_profiles_payload = {
    "n_clusters": int(N_CLUSTERS),
    "_note": "Auto-regenerated by 07_composite_treeshap.ipynb. Labels derived from per-cluster mean stats; edit `_heuristic_label` for tighter wording.",
    "clusters": profiles_clusters,
}

# Write to both notebook + backend artifact dirs (same pattern as the SHAP JSON).
profiles_path_local = OUT_DIR / "cluster_profiles.json"
with open(profiles_path_local, "w") as f:
    json.dump(cluster_profiles_payload, f, indent=2)
print("Saved", profiles_path_local)

profiles_path_backend = backend_xai / "cluster_profiles.json"
with open(profiles_path_backend, "w") as f:
    json.dump(cluster_profiles_payload, f, indent=2)
print("Saved", profiles_path_backend)

print("\nCluster profiles:")
for cid, p in profiles_clusters.items():
    print(f"  Cluster {cid}: {p['label']}")
    print(f"    {p['summary']}")
    print(f"    Top towns: {', '.join(p['top_towns'][:3])}")

## 8 — Summary

Composite TreeSHAP provides **fast** (~5ms/row) SHAP values for the hybrid ensemble by:
1. Computing TreeSHAP for XGBoost, LightGBM, and RandomForest
2. Weighting by the Ridge meta-learner coefficients

**Trade-offs:**
- Ignores Ridge component (~5% of signal typically)
- Assumes linear combination of tree predictions
- Slight deviation from "true" hybrid SHAP (validated against KernelSHAP)

**Artifacts created:**
- `composite_treeshap_config.json` — Explainer configuration and meta coefficients
- `composite_treeshap_values.joblib` — Computed SHAP values for sample
- `composite_treeshap_global_importance.json` — Global and per-cluster importance
- `composite_treeshap_summary_*.png` — Visualization plots

**Use cases:**
- Production explanations (fast enough for real-time)
- Batch processing of many predictions
- Per-cluster analysis of feature importance

In [27]:
print("\n── Composite TreeSHAP Artifacts ──")
for p in sorted(OUT_DIR.glob("composite_treeshap*")):
    if p.is_file():
        print(f"  {p.name}: {p.stat().st_size/1024:,.1f} KB")


── Composite TreeSHAP Artifacts ──
  composite_treeshap_config.json: 3.3 KB
  composite_treeshap_global_importance.json: 15.2 KB
  composite_treeshap_summary_bar.png: 74.2 KB
  composite_treeshap_summary_beeswarm.png: 148.6 KB
  composite_treeshap_values.joblib: 564.6 KB
  composite_treeshap_waterfall_sample0.png: 114.5 KB
